In [1]:
import pandas as pd
import numpy as np
import os
import time
import random
random.seed(67)
np.random.seed(67)

# DATASET EXPLORATION AND EDA
here I explore the data set, it's structure and perform eda.

In [2]:
s=""
with open("/kaggle/input/datasets/melikechan/cifar100/cifar100/label_names.txt","r") as f:
    for line in f:
        s+=line
all_labels=s.strip().split('\n')

In [3]:
#seeing what files there are

total={}
for j in all_labels:
    x=os.listdir(f"/kaggle/input/datasets/melikechan/cifar100/cifar100/train/{j}")
    for i in x:
        full=os.path.splitext(f"/kaggle/input/datasets/melikechan/cifar100/cifar100/train/{j}/{i}")[1]
        total[full]=total.get(full,0)+1
print(total)

{'.png': 50000}


# TRANSFORMATIONS AND SETUP

In [4]:
import torchvision as tv
from torchvision import transforms as tr

In [5]:
m=(0.5071, 0.4867, 0.4408)
s=(0.2675, 0.2565, 0.2761)
traint=tr.Compose(
    [
        tr.RandomCrop(32,padding=4), #because cifar are 32x32 images 
        tr.RandomHorizontalFlip(),
        tr.Resize((224,224)),
        tr.ToTensor(),
        tr.Normalize(mean=m,std=s)
    ]
)
testt=tr.Compose(
    [
        tr.Resize((224,224)),
        tr.ToTensor(),
        tr.Normalize(mean=m,std=s)
    ]
)

In [6]:
traind=tv.datasets.ImageFolder("/kaggle/input/datasets/melikechan/cifar100/cifar100/train",transform=traint)
testd=tv.datasets.ImageFolder("/kaggle/input/datasets/melikechan/cifar100/cifar100/test",transform=testt)

In [7]:
x,y=traind[0]
print("Checking all labels and class labels are same: ",end="")
print(traind.classes==all_labels)
print("Length of train",len(traind))
print("Length of test",len(testd))

Checking all labels and class labels are same: True
Length of train 50000
Length of test 10000


## breaking down the dataset into parts

In [8]:
import random
import torch

In [9]:
random.seed(67) #for reproduce
ind=list(range(len(traind)))
random.shuffle(ind)
print("So here we take all the indices from 0 to 50000 and then we randomly shuffle it\n"
      +"we will then take 10% 25% 50% and 100% of the data to analyse the performance")

So here we take all the indices from 0 to 50000 and then we randomly shuffle it
we will then take 10% 25% 50% and 100% of the data to analyse the performance


In [10]:
d10=torch.utils.data.Subset(traind,ind[:5000]) # 10% data
d25=torch.utils.data.Subset(traind,ind[:12500]) # 25% data
d50=torch.utils.data.Subset(traind,ind[:25000]) # 50% data
d100=torch.utils.data.Subset(traind,ind) #all data

## DATA LOADER

In [11]:
from torch.utils.data import DataLoader as dl

train_loader_d10=dl(d10,batch_size=64,shuffle=True)
train_loader_d25=dl(d25,batch_size=64,shuffle=True)
train_loader_d50=dl(d50,batch_size=64,shuffle=True)
train_loader_full=dl(d100,batch_size=64,shuffle=True)

test_loader=dl(testd,batch_size=64,shuffle=False)

In [12]:
#inspection
images, labels = next(iter(train_loader_d10))
print("Images shape is",images.shape)
print("Labels shape is",labels.shape)

Images shape is torch.Size([64, 3, 224, 224])
Labels shape is torch.Size([64])


In [13]:
tepoch=10
dev=torch.device("cuda")
torch.cuda.manual_seed(67)
torch.cuda.manual_seed_all(67)

# TEACHER INITIALIZATION and TRAINING

## Setup

In [14]:
import torch.nn.functional as F
import torch as tor

In [15]:
teach=tv.models.resnet18(weights=None)
teach.fc=torch.nn.Linear(teach.fc.in_features,100)
teach=teach.to(dev)
criterion=torch.nn.CrossEntropyLoss()
optimizer=torch.optim.AdamW(teach.parameters(),lr=0.0003)

## Teacher training

In [16]:
start=time.time()

In [17]:
for epoch in range(tepoch):
    teach.train()
    loss_s=0
    for i,l in train_loader_full:
        i=i.to(dev)
        l=l.to(dev)
        optimizer.zero_grad()
        outputs = teach(i)
        loss=criterion(outputs,l)
        loss.backward()
        optimizer.step()
        loss_s+=loss.item()
    print(f"Epoch {epoch} training done")
    print("Average loss across batch =",loss_s/len(train_loader_full))
    teach.eval()
    cor=0
    tot=0
    with torch.no_grad():
        for i,l in test_loader:
            i=i.to(dev)
            l=l.to(dev)
            outputs=teach(i)
            preds=outputs.argmax(dim=1)
            cor+=(l==preds).sum().item()
            tot+=l.size(0)
        acc=cor/tot
    print(f"Epoch {epoch} evaluation done\nAccuracy = {acc}\n")
    

Epoch 0 training done
Average loss across batch = 3.6887500575741234
Epoch 0 evaluation done
Accuracy = 0.1615

Epoch 1 training done
Average loss across batch = 2.8942039540356688
Epoch 1 evaluation done
Accuracy = 0.306

Epoch 2 training done
Average loss across batch = 2.3553374152049384
Epoch 2 evaluation done
Accuracy = 0.4057

Epoch 3 training done
Average loss across batch = 2.00147783649547
Epoch 3 evaluation done
Accuracy = 0.4247

Epoch 4 training done
Average loss across batch = 1.7539930977784763
Epoch 4 evaluation done
Accuracy = 0.505

Epoch 5 training done
Average loss across batch = 1.5677282205018241
Epoch 5 evaluation done
Accuracy = 0.5369

Epoch 6 training done
Average loss across batch = 1.4113195312145117
Epoch 6 evaluation done
Accuracy = 0.5483

Epoch 7 training done
Average loss across batch = 1.2917115368958934
Epoch 7 evaluation done
Accuracy = 0.5717

Epoch 8 training done
Average loss across batch = 1.186977925355477
Epoch 8 evaluation done
Accuracy = 0.605

In [18]:
end=time.time()
print(f"Time take is {end-start} seconds")

Time take is 3764.160523891449 seconds


In [19]:
teach.eval()
print("Teacher locked")

Teacher locked


# MODEL TRAINING

## 10% data

In [20]:
import timm as tim
model=tim.create_model("vit_tiny_patch16_224", pretrained=False,num_classes=100)
model=model.to(dev)
optimizer=tor.optim.AdamW(model.parameters(),lr=0.0003)

In [21]:
start=time.time()

In [22]:
for epoch in range(tepoch):
    model.train()
    loss_s=0
    for i,l in train_loader_d10:
        i=i.to(dev)
        l=l.to(dev)
        optimizer.zero_grad()
        with torch.no_grad():
            teach_log=teach(i)
        outputs = model(i)
        hard_l=criterion(outputs,l)
        teach_probs=F.softmax(teach_log/4,dim=1)
        stud_probs=F.log_softmax(outputs/4,dim=1)
        soft_l=F.kl_div(stud_probs,teach_probs,reduction="batchmean")*16
        loss=0.6*hard_l+0.4*soft_l
        loss.backward()
        optimizer.step()
        loss_s+=loss.item()
    print(f"Epoch {epoch} training done")
    print("Average loss across batch =",loss_s/len(train_loader_d10))
    model.eval()
    cor=0
    tot=0
    with torch.no_grad():
        for i,l in test_loader:
            i=i.to(dev)
            l=l.to(dev)
            outputs=model(i)
            preds=outputs.argmax(dim=1)
            cor+=(l==preds).sum().item()
            tot+=l.size(0)
        acc=cor/tot
    print(f"Epoch {epoch} evaluation done\nAccuracy = {acc}\n")

Epoch 0 training done
Average loss across batch = 5.86727595027489
Epoch 0 evaluation done
Accuracy = 0.0419

Epoch 1 training done
Average loss across batch = 5.556169534031349
Epoch 1 evaluation done
Accuracy = 0.0562

Epoch 2 training done
Average loss across batch = 5.329698755771299
Epoch 2 evaluation done
Accuracy = 0.0681

Epoch 3 training done
Average loss across batch = 5.140418342397183
Epoch 3 evaluation done
Accuracy = 0.0737

Epoch 4 training done
Average loss across batch = 4.969916844669776
Epoch 4 evaluation done
Accuracy = 0.1019

Epoch 5 training done
Average loss across batch = 4.823142154307305
Epoch 5 evaluation done
Accuracy = 0.1117

Epoch 6 training done
Average loss across batch = 4.705303318892853
Epoch 6 evaluation done
Accuracy = 0.1148

Epoch 7 training done
Average loss across batch = 4.583835390549671
Epoch 7 evaluation done
Accuracy = 0.1229

Epoch 8 training done
Average loss across batch = 4.5534959805162645
Epoch 8 evaluation done
Accuracy = 0.131

Ep

In [23]:
end=time.time()
print(f"Time take is {end-start} seconds")

Time take is 728.0537929534912 seconds


In [24]:
torch.save(
    model.state_dict(),
    "vit_10pct.pth"
)

## 25% data

In [25]:
model=tim.create_model("vit_tiny_patch16_224", pretrained=False,num_classes=100)
model=model.to(dev)
optimizer=tor.optim.AdamW(model.parameters(),lr=0.0003)

In [26]:
start=time.time()

In [27]:
for epoch in range(tepoch):
    model.train()
    loss_s=0
    for i,l in train_loader_d25:
        i=i.to(dev)
        l=l.to(dev)
        optimizer.zero_grad()
        with torch.no_grad():
            teach_log=teach(i)
        outputs = model(i)
        hard_l=criterion(outputs,l)
        teach_probs=F.softmax(teach_log/4,dim=1)
        stud_probs=F.log_softmax(outputs/4,dim=1)
        soft_l=F.kl_div(stud_probs,teach_probs,reduction="batchmean")*16
        loss=0.6*hard_l+0.4*soft_l
        loss.backward()
        optimizer.step()
        loss_s+=loss.item()
    print(f"Epoch {epoch} training done")
    print("Average loss across batch =",loss_s/len(train_loader_d25))
    model.eval()
    cor=0
    tot=0
    with torch.no_grad():
        for i,l in test_loader:
            i=i.to(dev)
            l=l.to(dev)
            outputs=model(i)
            preds=outputs.argmax(dim=1)
            cor+=(l==preds).sum().item()
            tot+=l.size(0)
        acc=cor/tot
    print(f"Epoch {epoch} evaluation done\nAccuracy = {acc}\n")

Epoch 0 training done
Average loss across batch = 5.644023423292199
Epoch 0 evaluation done
Accuracy = 0.0646

Epoch 1 training done
Average loss across batch = 5.027419608466479
Epoch 1 evaluation done
Accuracy = 0.1077

Epoch 2 training done
Average loss across batch = 4.69246123031694
Epoch 2 evaluation done
Accuracy = 0.1365

Epoch 3 training done
Average loss across batch = 4.460042715072632
Epoch 3 evaluation done
Accuracy = 0.1572

Epoch 4 training done
Average loss across batch = 4.280986523141666
Epoch 4 evaluation done
Accuracy = 0.1717

Epoch 5 training done
Average loss across batch = 4.114764686749906
Epoch 5 evaluation done
Accuracy = 0.1975

Epoch 6 training done
Average loss across batch = 3.960614788288973
Epoch 6 evaluation done
Accuracy = 0.2093

Epoch 7 training done
Average loss across batch = 3.8443243406256853
Epoch 7 evaluation done
Accuracy = 0.2097

Epoch 8 training done
Average loss across batch = 3.717563876083919
Epoch 8 evaluation done
Accuracy = 0.2319

E

In [28]:
end=time.time()
print(f"Time take is {end-start} seconds")

Time take is 1236.3117957115173 seconds


In [29]:
torch.save(
    model.state_dict(),
    "vit_25pct.pth"
)

## 50% data

In [30]:
model=tim.create_model("vit_tiny_patch16_224", pretrained=False,num_classes=100)
model=model.to(dev)
optimizer=tor.optim.AdamW(model.parameters(),lr=0.0003)

In [31]:
start=time.time()

In [32]:
for epoch in range(tepoch):
    model.train()
    loss_s=0
    for i,l in train_loader_d50:
        i=i.to(dev)
        l=l.to(dev)
        optimizer.zero_grad()
        with torch.no_grad():
            teach_log=teach(i)
        outputs = model(i)
        hard_l=criterion(outputs,l)
        teach_probs=F.softmax(teach_log/4,dim=1)
        stud_probs=F.log_softmax(outputs/4,dim=1)
        soft_l=F.kl_div(stud_probs,teach_probs,reduction="batchmean")*16
        loss=0.6*hard_l+0.4*soft_l
        loss.backward()
        optimizer.step()
        loss_s+=loss.item()
    print(f"Epoch {epoch} training done")
    print("Average loss across batch =",loss_s/len(train_loader_d50))
    model.eval()
    cor=0
    tot=0
    with torch.no_grad():
        for i,l in test_loader:
            i=i.to(dev)
            l=l.to(dev)
            outputs=model(i)
            preds=outputs.argmax(dim=1)
            cor+=(l==preds).sum().item()
            tot+=l.size(0)
        acc=cor/tot
    print(f"Epoch {epoch} evaluation done\nAccuracy = {acc}\n")

Epoch 0 training done
Average loss across batch = 5.372746057827454
Epoch 0 evaluation done
Accuracy = 0.1048

Epoch 1 training done
Average loss across batch = 4.64953962067509
Epoch 1 evaluation done
Accuracy = 0.1388

Epoch 2 training done
Average loss across batch = 4.293255450475551
Epoch 2 evaluation done
Accuracy = 0.1679

Epoch 3 training done
Average loss across batch = 4.0200927385588745
Epoch 3 evaluation done
Accuracy = 0.188

Epoch 4 training done
Average loss across batch = 3.809897138639484
Epoch 4 evaluation done
Accuracy = 0.2357

Epoch 5 training done
Average loss across batch = 3.5792633822506956
Epoch 5 evaluation done
Accuracy = 0.2574

Epoch 6 training done
Average loss across batch = 3.4052057668681033
Epoch 6 evaluation done
Accuracy = 0.2722

Epoch 7 training done
Average loss across batch = 3.2074477190861614
Epoch 7 evaluation done
Accuracy = 0.307

Epoch 8 training done
Average loss across batch = 3.041776431491003
Epoch 8 evaluation done
Accuracy = 0.3285



In [33]:
end=time.time()
print(f"Time take is {end-start} seconds")

Time take is 2260.760406732559 seconds


In [34]:
torch.save(
    model.state_dict(),
    "vit_50pct.pth"
)

## all data

In [35]:
model=tim.create_model("vit_tiny_patch16_224", pretrained=False,num_classes=100)
model=model.to(dev)
optimizer=tor.optim.AdamW(model.parameters(),lr=0.0003)

In [36]:
for epoch in range(tepoch):
    model.train()
    loss_s=0
    for i,l in train_loader_full:
        i=i.to(dev)
        l=l.to(dev)
        optimizer.zero_grad()
        with torch.no_grad():
            teach_log=teach(i)
        outputs = model(i)
        hard_l=criterion(outputs,l)
        teach_probs=F.softmax(teach_log/4,dim=1)
        stud_probs=F.log_softmax(outputs/4,dim=1)
        soft_l=F.kl_div(stud_probs,teach_probs,reduction="batchmean")*16
        loss=0.6*hard_l+0.4*soft_l
        loss.backward()
        optimizer.step()
        loss_s+=loss.item()
    print(f"Epoch {epoch} training done")
    print("Average loss across batch =",loss_s/len(train_loader_full))
    model.eval()
    cor=0
    tot=0
    with torch.no_grad():
        for i,l in test_loader:
            i=i.to(dev)
            l=l.to(dev)
            outputs=model(i)
            preds=outputs.argmax(dim=1)
            cor+=(l==preds).sum().item()
            tot+=l.size(0)
        acc=cor/tot
    print(f"Epoch {epoch} evaluation done\nAccuracy = {acc}\n")

Epoch 0 training done
Average loss across batch = 4.990380712177442
Epoch 0 evaluation done
Accuracy = 0.1431

Epoch 1 training done
Average loss across batch = 4.108944132809749
Epoch 1 evaluation done
Accuracy = 0.2164

Epoch 2 training done
Average loss across batch = 3.6639474645599988
Epoch 2 evaluation done
Accuracy = 0.2568

Epoch 3 training done
Average loss across batch = 3.3121075575309034
Epoch 3 evaluation done
Accuracy = 0.3055

Epoch 4 training done
Average loss across batch = 3.010810691377391
Epoch 4 evaluation done
Accuracy = 0.3262

Epoch 5 training done
Average loss across batch = 2.766511390276272
Epoch 5 evaluation done
Accuracy = 0.3839

Epoch 6 training done
Average loss across batch = 2.568196773833936
Epoch 6 evaluation done
Accuracy = 0.4062

Epoch 7 training done
Average loss across batch = 2.4099073504547937
Epoch 7 evaluation done
Accuracy = 0.4233

Epoch 8 training done
Average loss across batch = 2.2627384408050792
Epoch 8 evaluation done
Accuracy = 0.450

In [37]:
torch.save(
    model.state_dict(),
    "vit_100pct.pth"
)